<a href="https://colab.research.google.com/github/Valdes-Tresanco-MS/gmx_MMPBSA/blob/colab-notebook/notebooks/gmx_MMPBSA_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# gmx_MMPBSA examples on Google Colab CPUs

This notebook installs a CPU-only gmx_MMPBSA environment in Google Colab, runs selected bundled examples, and displays the generated result files. It is intended as a quick runtime check and a starting point for adapting commands to your own systems.

Use **Runtime > Change runtime type > CPU** before running the notebook. A fresh Colab runtime is recommended.

## Runtime notes

gmx_MMPBSA currently requires Python 3.11. Colab runtime Python versions change over time, so this notebook creates an isolated conda environment in `/content/miniforge3` and runs gmx_MMPBSA commands through `conda run`. The notebook kernel can stay on Colab's default Python.

In [ ]:
import os
import platform
import os
import multiprocessing
from pathlib import Path

print("Python kernel:", platform.python_version())
print("Platform:", platform.platform())
print("CPU cores:", multiprocessing.cpu_count())
print("Working directory:", Path.cwd())

## 1. Install gmx_MMPBSA and dependencies

This cell installs Miniforge if needed, creates a `gmxMMPBSA` conda environment, installs AmberTools, GROMACS, MPI support, and installs gmx_MMPBSA from PyPI. It can take several minutes on a fresh Colab runtime.

In [ ]:
%%bash
set -euo pipefail

log() {
  echo
  echo "[$(date +%H:%M:%S)] $*"
}

MINIFORGE=/content/miniforge3
CONDA="$MINIFORGE/bin/conda"

log "Starting gmx_MMPBSA Colab environment setup"
echo "Install prefix: $MINIFORGE"

if [ ! -x "$CONDA" ]; then
  log "Downloading Miniforge installer"
  wget --progress=dot:giga https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh -O /tmp/miniforge.sh
  log "Installing Miniforge"
  bash /tmp/miniforge.sh -b -p "$MINIFORGE"
else
  log "Reusing existing Miniforge installation"
fi

log "Configuring conda channel priority"
"$CONDA" config --set channel_priority strict

if ! "$CONDA" env list | awk '{print $1}' | grep -qx gmxMMPBSA; then
  log "Creating gmxMMPBSA environment"
  echo "This is the longest step. Conda is solving and installing AmberTools, GROMACS, MPI, and Python packages."
  "$CONDA" create -n gmxMMPBSA -c conda-forge -y \
    python=3.11 pip git \
    'ambertools<24' 'gromacs<2026' \
    mpi4py=4.0.1 numpy=1.26.4 scipy=1.14.1 \
    pandas=1.5.3 matplotlib=3.7.3 seaborn=0.11.2 \
    'parmed>=4.2.2' tqdm
else
  log "Reusing existing gmxMMPBSA conda environment"
fi

log "Installing or updating gmx_MMPBSA from PyPI"
"$CONDA" run -n gmxMMPBSA python -m pip install -U gmx_MMPBSA

log "Installed command versions"
"$CONDA" run -n gmxMMPBSA python --version
"$CONDA" run -n gmxMMPBSA gmx_MMPBSA --version
"$CONDA" run -n gmxMMPBSA gmx --version | sed -n '1,4p'

log "Cleaning conda package cache"
"$CONDA" clean -afy

log "Environment setup finished"

## 2. Verify installed tools

In [ ]:
%%bash
set -euo pipefail
CONDA=/content/miniforge3/bin/conda
ENV_PREFIX=/content/miniforge3/envs/gmxMMPBSA

"$CONDA" run -n gmxMMPBSA python --version
"$CONDA" run -n gmxMMPBSA gmx_MMPBSA --version

if [ -x "$ENV_PREFIX/bin/amber_MMPBSA" ]; then
  "$ENV_PREFIX/bin/amber_MMPBSA" --version
else
  echo "amber_MMPBSA command not found; skipping AMBER-only example test 25."
fi

"$CONDA" run -n gmxMMPBSA gmx --version | sed -n '1,8p'
"$CONDA" run -n gmxMMPBSA mpirun --version | sed -n '1,4p'

## 3. Run bundled examples

The default runs the protein-ligand single-trajectory example (`-t 3`) with 2 MPI ranks. You can change `TEST_SELECTION`, `NUM_PROCESSORS`, and `NUM_CONCURRENT` before running the next cell.

Avoid setting `NUM_PROCESSORS * NUM_CONCURRENT` higher than the CPU count reported above.

In [ ]:
# gmx_MMPBSA_test selections:
# 3 = Protein-Ligand (Single trajectory approximation)
# 2 = Fast set, 1 = Minimal set, 0 = All examples
# 25 = AMBER input files; use only if amber_MMPBSA is available in the verify cell
TEST_SELECTION = "3"
NUM_PROCESSORS = 2
NUM_CONCURRENT = 1
WORKDIR = "/content"

In [ ]:
import multiprocessing
import shlex
import subprocess

cpu_count = multiprocessing.cpu_count()
requested = int(NUM_PROCESSORS) * int(NUM_CONCURRENT)
if requested > cpu_count:
    print(f"Warning: requested up to {requested} MPI ranks on {cpu_count} CPU cores.")

cmd = [
    "/content/miniforge3/bin/conda", "run", "-n", "gmxMMPBSA",
    "gmx_MMPBSA_test", "-f", WORKDIR, "-ng",
    "-n", str(NUM_PROCESSORS), "-j", str(NUM_CONCURRENT),
    "-t", *shlex.split(str(TEST_SELECTION)),
]
print("Running:", " ".join(shlex.quote(part) for part in cmd))
env = os.environ.copy()
env["OMPI_ALLOW_RUN_AS_ROOT"] = "1"
env["OMPI_ALLOW_RUN_AS_ROOT_CONFIRM"] = "1"

result = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env=env)
print(result.stdout)
if result.returncode:
    raise RuntimeError(f"gmx_MMPBSA_test failed with exit code {result.returncode}")

## 4. Inspect result files

In [ ]:
from pathlib import Path

base = Path(WORKDIR) / "gmx_MMPBSA_test" / "examples"
result_files = sorted(base.rglob("FINAL_RESULTS_MMPBSA.dat"))
csv_files = sorted(base.rglob("FINAL_RESULTS_MMPBSA.csv"))
log_files = sorted(base.rglob("*.log"))

print("Result text files:")
for path in result_files:
    print(" -", path)

print("\nResult CSV files:")
for path in csv_files:
    print(" -", path)

print("\nLog files:")
for path in log_files:
    print(" -", path)

In [ ]:
if result_files:
    print(result_files[0])
    print(result_files[0].read_text(errors="replace")[:8000])
else:
    print("No FINAL_RESULTS_MMPBSA.dat files found.")

In [ ]:
import pandas as pd

if csv_files:
    df = pd.read_csv(csv_files[0])
    display(df.head())
else:
    print("No FINAL_RESULTS_MMPBSA.csv files found.")

## Next step: adapt to your own files

After this example run works, use the command printed by each example README as the template for your own data. In a future upload-oriented version, this notebook can add file upload or Google Drive mounting cells and build the `gmx_MMPBSA` command from user-provided file paths.